# 06 · Can a pretrained semantic model improve our tag suggestions?

**DR-18 question:** Can a model that learned language relationships elsewhere suggest useful topics from a demonstration title, without training on our historical tags?

We test **S1: multilingual mDeBERTa**, the first semantic candidate named in the ADR. This is a natural language inference (**NLI**) model: it reads two texts and scores whether the first supports, contradicts, or leaves the second uncertain. Here the first text is an event title; the second is a sentence describing a tag.

For the synthetic title **Sichere Radwege jetzt**, we ask whether the title supports a sentence about transport and road safety. We also ask separately about housing, religion, and every other approved topic. Several topics can apply to the same title.

**Zero-shot here means no fitting on DemoLogic training labels.** The model was already trained by its publisher. We supply fixed tag descriptions and choose an output cutoff on development examples. That cutoff selection is still development work; zero-shot does not mean we can skip evaluation. This experiment uses NLI, not a sentence-embedding classifier; embeddings are an optional later comparison.

## What stays comparable?

- Event input: the same original titles and saved event-group representatives as Notebooks 03 and 04.
- Development: 188 validation representatives, split into 94 for cutoff selection and 94 for checking it, with the same seed and order.
- Output: up to two tags, plus exact ties at second place; also show the strict two-tag policy.
- References: historical validation tags and the already reviewed 40-record development pilot. The final-test answers are not used for fitting, tuning, or scoring.

**Information differs by method:** TF-IDF learned from 732 historical training examples. NLI uses pretrained weights and the fixed tag descriptions below. Neither receives extra location fields, research, or model-generated evidence. Comparing them tests these complete tagging methods; it does not isolate architecture alone.

The notebook explains each step, shows the scores and errors, and saves a private report. No inference API is called. Downloaded weights and event scores stay in ignored local folders. Clear notebook outputs before committing.

## 1. Environment and model choice

Inside the existing Jupyter container, install the optional dependency group before running this notebook:

```bash
export UV_PROJECT_ENVIRONMENT=/home/jovyan/.venvs/demo-logic
uv sync --locked --group semantic
```

Restart **DemoLogic (uv)** afterward. This group includes the earlier notebook dependencies plus PyTorch, Transformers, and SentencePiece. CI uses the lighter notebook group and synthetic tests; it does not download this model or access private data.

We pin `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` at revision `8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c`. The publisher lists an MIT license. The selected checkpoint and tokenizer files total about **562 MB**; Python dependencies need additional disk space. We run CPU inference in float32 with small batches. Actual process memory is measured later and exceeds the on-disk checkpoint size.

Sources checked on 28 September 2026: [pinned model card](https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-mnli-xnli/blob/8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c/README.md) and [Hugging Face zero-shot classification documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines#transformers.ZeroShotClassificationPipeline). The publisher's NLI benchmark scores are not expected DemoLogic tagging scores.

In [ ]:
import json
import os
import platform
import resource
import sys
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
from huggingface_hub import snapshot_download
from huggingface_hub.errors import LocalEntryNotFoundError
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoModelForSequenceClassification, AutoTokenizer

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook inside the DemoLogic repository")
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root))

from notebooks.tools import (
    aligned_predictions,
    comparison_row,
    file_sha256,
    fingerprint,
    matching_report,
    notebook_sha256,
    percent,
    verify_sha256,
)
from notebooks.tools import show_table as show

In [ ]:
from demo_logic.benchmark import _csv, load_benchmark, representative_selection
from demo_logic.evaluation import multilabel_metrics, parse_review_tags
from demo_logic.notebook_support import save_private_json
from demo_logic.snapshot import load_snapshot
from demo_logic.taxonomy import topic_tags
from notebooks.tools.review import write_nli_review

## 2. Load the same development examples

We verify the source snapshot and saved benchmark. Related editions stay in their existing split. We select one representative per event group using DR-40's saved rule. Training records are counted for comparison but are not used to fit this NLI model.

The snapshot loader sees all source records to verify the saved split. Only validation representatives are sent to the scorer. Pilot answers are read later as a development diagnostic, never as inference input. This cell does not open the final-review CSV to score predictions.

In [ ]:
rows, source = load_snapshot(root / "data/dataset.jsonl", root / "data/manifest.json")
bundle = load_benchmark(root / "data/benchmark", rows, source["content_sha256"])
if bundle.manifest.get("status") != "frozen" or source["taxonomy_version"] != "v2":
    raise ValueError("Notebook 06 requires locked split membership and taxonomy v2")
selection = representative_selection(bundle)
selection_path = save_private_json(
    root / "data/benchmark" / f"{selection['version']}.json", selection
)
selected_ids = {item["external_id"] for item in selection["records"]}
by_id = {row["external_id"]: row for row in rows}
validation = [
    {**by_id[item["external_id"]], "group_id": item["group_id"]}
    for item in bundle.manifest["assignments"]
    if item["split"] == "validation" and item["external_id"] in selected_ids
]
training_count = sum(
    item["split"] == "training" and item["external_id"] in selected_ids
    for item in bundle.manifest["assignments"]
)
tags = sorted(topic_tags(source["taxonomy_version"]))
reviewed_pilot = {
    row["external_id"]: parse_review_tags(row["reviewed_tags"], set(tags))
    for row in _csv(bundle.paths["rubric_pilot"], {"external_id", "reviewed_tags"})
}
if len(reviewed_pilot) != 40 or not set(reviewed_pilot).issubset(
    {row["external_id"] for row in validation}
):
    raise ValueError("Need all 40 reviewed development examples in validation")
RANDOM_SEED = 42
splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_SEED)
groups = [row["group_id"] for row in validation]
tune_index, check_index = next(splitter.split(validation, groups=groups))
assert not ({groups[i] for i in tune_index} & {groups[i] for i in check_index})
pilot_index = [i for i, row in enumerate(validation) if row["external_id"] in reviewed_pilot]
historical = [row["tag_slugs"] for row in validation]
expected_fingerprints = {
    str(selection_path.relative_to(root)): file_sha256(selection_path),
    "data/manifest.json": file_sha256(root / "data/manifest.json"),
    "data/benchmark/split-manifest.json": file_sha256(bundle.paths["manifest"]),
    "data/benchmark/rubric-pilot-queue.csv": file_sha256(bundle.paths["rubric_pilot"]),
}
show(
    ["Input", "Count"],
    [
        ("Training representatives (not used to fit NLI)", training_count),
        ("Choose the cutoff", len(tune_index)),
        ("Check the chosen cutoff", len(check_index)),
        ("Reviewed pilot, already part of validation", len(pilot_index)),
        ("Final-test records scored", 0),
    ],
)

## 3. Turn the tag definitions into questions

Each German sentence below is a **hypothesis**: a statement that the title may or may not support. These are fixed summaries of rubric-v1's topic definitions. They are written before looking at this model's scores; we do not rewrite them in response to check-set errors.

For example: title **Sichere Radwege jetzt** + a sentence about transport. The model returns scores for support, contradiction, and uncertainty. We repeat this for each tag independently, so tags do not compete for a total of 100%.

These short sentences cannot enforce every exclusion and overlap rule in the rubric. For example, a model may treat an LGBTQ+ demand as a general human-rights demand even when the rubric requires a separate rights issue. That is a limitation to measure, not a reason to alter reference tags.

In [ ]:
HYPOTHESES = {
    "environment": "Das zentrale Thema dieser Versammlung ist Klima, Energiewende, Natur-, Tier- oder Umweltschutz.",
    "mobility": "Das zentrale Thema dieser Versammlung ist Verkehr, Radfahren, Fußverkehr, öffentlicher Nahverkehr oder sichere Straßen.",
    "housing": "Das zentrale Thema dieser Versammlung ist Miete, Wohnraum, Verdrängung, Stadtentwicklung, öffentlicher Raum oder die Entwicklung eines Wohnviertels.",
    "work": "Das zentrale Thema dieser Versammlung ist Arbeit, Lohn, Arbeitsbedingungen, Streik, Gewerkschaften oder Tarifverhandlungen.",
    "social-issues": "Das zentrale Thema dieser Versammlung ist Armut, Sozialleistungen, Obdachlosigkeit, Gesundheitsversorgung, Pflege oder Schutz vor Gewalt.",
    "education": "Das zentrale Thema dieser Versammlung ist Bildung, Schule, Hochschule, Kita, Forschung oder Zugang zu Bildung.",
    "lgbtq": "Das zentrale Thema dieser Versammlung sind queere Menschen, LGBTQ-Rechte, Pride oder die Anliegen von trans, inter, lesbischen, schwulen oder bisexuellen Menschen.",
    "international-politics": "Das zentrale Thema dieser Versammlung ist Politik anderer Länder, internationale Beziehungen, Außenpolitik, Krieg, Frieden oder politische Unterdrückung im Ausland.",
    "human-rights": "Das zentrale Thema dieser Versammlung ist Menschenrechte, Bürgerrechte, Antirassismus, Migration, Gleichberechtigung, Barrierefreiheit oder Inklusion.",
    "politics": "Das zentrale Thema dieser Versammlung ist deutsche Innenpolitik: Wahlen, Parteien, staatliche Institutionen, demokratische Verfahren oder politische Vertretung in Deutschland.",
    "culture": "Das zentrale Thema dieser Versammlung ist Kunst, Kultur, Medien, Geschichte, Gedenken, kulturelle Einrichtungen oder Sport.",
    "religion": "Das zentrale Thema dieser Versammlung ist Religion, Glauben, Religionsgemeinschaften, Gebete, Gottesdienste oder religiöse Einrichtungen.",
}
if set(HYPOTHESES) != set(tags):
    raise ValueError("Each approved tag must have exactly one hypothesis")
rubric_version = "rubric-v1"
EXPECTED_RUBRIC_SHA256 = "271f4b5ef0680df56481a3579e3871772df60da3b0c0a58156bfe83b8e8bba11"
rubric_sha256 = verify_sha256(root / "docs/tagging-rubric.md", EXPECTED_RUBRIC_SHA256)
hypotheses_sha256 = fingerprint(HYPOTHESES)
show(["Tag", "Statement tested against the title"], [(tag, HYPOTHESES[tag]) for tag in tags])

## 4. Load the pinned model locally

The first run downloads only the selected public model files. Later runs use the same local snapshot. The checkpoint and tokenizer hashes are verified. We use float32 on CPU, four tag pairs per batch, and at most four CPU threads. No event text is sent to Hugging Face.

A pair can contain at most 512 tokens. If necessary, we shorten the event title, keeping the tag sentence intact. The report counts affected pairs and events so truncation is visible. We do not treat a download error, malformed score, or failed inference as a completed experiment.

In [ ]:
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
MODEL_REVISION = "8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c"
MODEL_LICENSE = "MIT"
MODEL_FILES = [
    "config.json",
    "model.safetensors",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "added_tokens.json",
    "spm.model",
    "README.md",
]
EXPECTED_WEIGHT_SHA256 = "65af59b1ff4450b09ecbf13ca35c840dbf038b26ff8e10e5ea89ca724828ed1e"
EXPECTED_TOKENIZER_SHA256 = "13c8d666d62a7bc4ac8f040aab68e942c861f93303156cc28f5c7e885d86d6e3"
BATCH_SIZE = 4
MAX_LENGTH = 512
CPU_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
torch.manual_seed(RANDOM_SEED)

started = perf_counter()
download_args = {
    "repo_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "allow_patterns": MODEL_FILES,
    "cache_dir": str(root / "artifacts/hf-cache"),
}
try:
    model_dir = Path(snapshot_download(**download_args, local_files_only=True))
except LocalEntryNotFoundError:
    model_dir = Path(snapshot_download(**download_args))
if any(not (model_dir / name).is_file() for name in MODEL_FILES):
    model_dir = Path(snapshot_download(**download_args))
model_files_bytes = sum((model_dir / name).stat().st_size for name in MODEL_FILES)
verify_sha256(model_dir / "model.safetensors", EXPECTED_WEIGHT_SHA256)
verify_sha256(model_dir / "spm.model", EXPECTED_TOKENIZER_SHA256)
snapshot_seconds = perf_counter() - started
started = perf_counter()
tokenizer = AutoTokenizer.from_pretrained(
    model_dir, use_fast=False, local_files_only=True, trust_remote_code=False
)
model = (
    AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        local_files_only=True,
        trust_remote_code=False,
        use_safetensors=True,
        dtype=torch.float32,
    )
    .to("cpu")
    .eval()
)
model_load_seconds = perf_counter() - started
label_ids = {name.lower(): int(index) for name, index in model.config.label2id.items()}
if set(label_ids) != {"entailment", "neutral", "contradiction"}:
    raise ValueError("Unexpected NLI output labels; stop rather than guessing their order")
show(
    ["Model fact", "Value"],
    [
        ("Revision", MODEL_REVISION),
        ("License", MODEL_LICENSE),
        ("Selected model files, MB (decimal)", round(model_files_bytes / 1_000_000, 1)),
        ("Snapshot/cache verification seconds", round(snapshot_seconds, 2)),
        ("Load model and tokenizer seconds", round(model_load_seconds, 2)),
        ("Device / precision / threads", f"CPU / float32 / {CPU_THREADS}"),
        ("PyTorch / Transformers", f"{torch.__version__} / {transformers.__version__}"),
    ],
)

## 5. Score each title against all 12 tags

We use the usual multi-label NLI score: compare **support (entailment)** with **contradiction**, separately for each tag. The neutral output is excluded from this two-way score and recorded separately. This follows the scoring idea used by Hugging Face's multi-label zero-shot pipeline.

A score of 0.80 is **not a verified 80% chance that we will accept the tag**. In particular, a high neutral output can be hidden by the support-versus-contradiction comparison. Our cutoff experiment measures whether these scores are useful on our task.

One private cache file stores each event's scores and measured inference time. Its key includes the exact title, snapshot, revision, tag sentences, software versions, and inference settings. A change creates a new cache entry. Completed entries can be reused after interruption; missing events are scored on the next run. Timings from reused entries are marked as recorded measurements, not the speed of a fresh rerun.

In [ ]:
def nli_tag_scores(logits, label_ids):
    """Return per-pair support-vs-contradiction scores and full neutral probabilities.

    Read the class indices from the pinned model configuration. Normalize each
    pair independently so multiple tags can score highly. Neutral is excluded
    from the tag score, matching multi-label zero-shot pipeline semantics;
    it is also returned for diagnostics. Reject malformed or non-finite output.
    """
    values = np.asarray(logits, dtype=np.float64)
    if (
        values.ndim != 2
        or values.shape[1] != 3
        or not np.isfinite(values).all()
        or set(label_ids) != {"entailment", "neutral", "contradiction"}
        or set(label_ids.values()) != {0, 1, 2}
    ):
        raise ValueError("Expected finite logits for three distinct named NLI classes")
    pair = values[:, [label_ids["contradiction"], label_ids["entailment"]]]
    pair = np.exp(pair - pair.max(axis=1, keepdims=True))
    score = pair[:, 1] / pair.sum(axis=1)
    full = np.exp(values - values.max(axis=1, keepdims=True))
    neutral = full[:, label_ids["neutral"]] / full.sum(axis=1)
    return score, neutral


hypothesis_text = [HYPOTHESES[tag] for tag in tags]
hypothesis_lengths = [
    len(tokenizer.encode(text, add_special_tokens=False)) for text in hypothesis_text
]
special_tokens = tokenizer.num_special_tokens_to_add(pair=True)


def score_title(title):
    """Score one event locally, keeping all tag hypotheses in catalog order.

    Time tokenization plus model inference for all tags. Count truncation from
    the original pair lengths. No tag references or event IDs enter the model.
    An inference error stops the notebook; successful earlier events stay cached.
    """
    started = perf_counter()
    title_length = len(tokenizer.encode(title, add_special_tokens=False))
    truncated_pairs = sum(
        title_length + length + special_tokens > MAX_LENGTH for length in hypothesis_lengths
    )
    scores, neutral_scores = [], []
    with torch.inference_mode():
        for start in range(0, len(tags), BATCH_SIZE):
            hypotheses = hypothesis_text[start : start + BATCH_SIZE]
            inputs = tokenizer(
                [title] * len(hypotheses),
                hypotheses,
                padding=True,
                truncation="only_first",
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            logits = model(**inputs).logits.cpu().numpy()
            tag_score, neutral = nli_tag_scores(logits, label_ids)
            scores.extend(tag_score.tolist())
            neutral_scores.extend(neutral.tolist())
    return {
        "scores": scores,
        "neutral_probabilities": neutral_scores,
        "prediction_seconds": perf_counter() - started,
        "truncated_pairs": truncated_pairs,
    }


# A synthetic warm-up avoids counting first-call initialization in event latency.
started = perf_counter()
synthetic_result = score_title("Sichere Radwege jetzt")
warmup_seconds = perf_counter() - started
show(
    ["Synthetic example tag", "NLI score", "Neutral output"],
    [
        (tag, round(score, 3), round(neutral, 3))
        for tag, score, neutral in zip(
            tags, synthetic_result["scores"], synthetic_result["neutral_probabilities"]
        )
    ],
)

In [ ]:
inference_settings = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "hypotheses": HYPOTHESES,
    "rubric_version": rubric_version,
    "rubric_sha256": rubric_sha256,
    "tags": tags,
    "dtype": "float32",
    "device": "cpu",
    "cpu_threads": CPU_THREADS,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "truncation": "only_first",
    "score_type": "entailment_vs_contradiction_softmax",
    "label_ids": label_ids,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "platform": platform.platform(),
    "python": platform.python_version(),
    "snapshot_sha256": source["content_sha256"],
    "selection_sha256": selection["selection_sha256"],
}
settings_sha256 = fingerprint(inference_settings)
cache_dir = root / "reports/semantic-cache" / settings_sha256
cache_hits = 0
scored_events = []
started = perf_counter()
for index, row in enumerate(validation):
    identity = {
        "settings_sha256": settings_sha256,
        "external_id": row["external_id"],
        "title_sha256": fingerprint(row["title"]),
    }
    cache_path = cache_dir / f"{fingerprint(identity)}.json"
    if cache_path.is_file():
        result = json.loads(cache_path.read_text(encoding="utf-8"))
        if result.get("identity") != identity:
            raise ValueError("Cache identity mismatch; do not use stale event scores")
        cache_hits += 1
    else:
        result = {"identity": identity, **score_title(row["title"])}
        save_private_json(cache_path, result)
    for key in ("scores", "neutral_probabilities"):
        values = np.asarray(result[key])
        if (
            values.shape != (len(tags),)
            or not np.isfinite(values).all()
            or np.any((values < 0) | (values > 1))
        ):
            raise ValueError(f"Invalid cached {key}; stop and inspect the private cache")
    scored_events.append(result)
    if (index + 1) % 20 == 0 or index + 1 == len(validation):
        print(f"Scored or loaded {index + 1}/{len(validation)} development events", flush=True)
scoring_wall_seconds = perf_counter() - started
scores = np.asarray([row["scores"] for row in scored_events])
event_seconds = np.asarray([row["prediction_seconds"] for row in scored_events])
show(
    ["Scoring fact", "Value"],
    [
        ("Development events / tag pairs", f"{len(validation)} / {len(validation) * len(tags)}"),
        ("Cache hits / fresh events", f"{cache_hits} / {len(validation) - cache_hits}"),
        ("This run: scoring/cache loop seconds", round(scoring_wall_seconds, 2)),
        ("Recorded inference seconds, summed over events", round(float(event_seconds.sum()), 2)),
        (
            "Recorded median / p95 seconds per event",
            f"{np.median(event_seconds):.3f} / {np.percentile(event_seconds, 95):.3f}",
        ),
        (
            "Events with at least one truncated pair",
            sum(row["truncated_pairs"] > 0 for row in scored_events),
        ),
    ],
)

## 6. Choose on 94 examples, check on the other 94

The cutoff is the minimum NLI score needed to suggest a tag. We try a fixed grid from 0.05 to 0.95, plus 0.975 and 0.99 because NLI scores can be close to one. This grid was set before scoring. These scores have a different meaning from logistic-regression scores; copying its cutoff would not be justified.

For each output policy, choose the cutoff with the highest micro F1 on the first 94 examples; break F1 ties by taking the higher cutoff. Apply it unchanged to the other 94. We also show agreement with all historical validation labels and with the reviewed pilot; neither is a fresh final-test estimate. All 188 attempts, including events receiving no tag, remain in their appropriate denominators.

In [ ]:
THRESHOLDS = [round(value / 100, 3) for value in range(5, 100, 5)] + [0.975, 0.99]


def select_tags(values, threshold, include_ties=True):
    """Keep qualifying tags in score order, with optional exact ties at second place.

    Scores remain unrounded during selection. A high cutoff may produce an
    empty recommendation, which is counted as an abstention in the metrics.
    """
    ranked = sorted(
        ((tag, float(value)) for tag, value in zip(tags, values) if value >= threshold),
        key=lambda item: (-item[1], item[0]),
    )
    selected = ranked[:2]
    if include_ties and len(ranked) > 2:
        selected += [item for item in ranked[2:] if item[1] == selected[-1][1]]
    return [tag for tag, _ in selected]


def score_indices(reference, predictions, indices):
    """Measure a named subset without dropping abstentions or truncating reference tags."""
    return multilabel_metrics(
        [reference[i] for i in indices], [predictions[i] for i in indices], tags
    )


policies = []
for include_ties in (True, False):
    trials = []
    for threshold in THRESHOLDS:
        predicted = [select_tags(row, threshold, include_ties) for row in scores]
        trials.append(
            {"threshold": threshold, "metrics": score_indices(historical, predicted, tune_index)}
        )
    best = max(
        trials,
        key=lambda r: (
            r["metrics"]["micro_f1"] if r["metrics"]["micro_f1"] is not None else -1,
            r["threshold"],
        ),
    )
    predicted = [select_tags(row, best["threshold"], include_ties) for row in scores]
    policies.append(
        {
            "policy": "include_cutoff_ties" if include_ties else "strict_two",
            "threshold": best["threshold"],
            "threshold_trials": trials,
            "tuning": best["metrics"],
            "group_heldout_check": score_indices(historical, predicted, check_index),
            "historical_agreement": multilabel_metrics(historical, predicted, tags),
            "reviewed_pilot": multilabel_metrics(
                [reviewed_pilot[validation[i]["external_id"]] for i in pilot_index],
                [predicted[i] for i in pilot_index],
                tags,
            ),
            "predictions": [
                {"external_id": row["external_id"], "predicted": pred}
                for row, pred in zip(validation, predicted)
            ],
        }
    )
primary = next(p for p in policies if p["policy"] == "include_cutoff_ties")
show(
    ["Policy", "Chosen cutoff", "Choose on", "Check on"],
    [(p["policy"], p["threshold"], len(tune_index), len(check_index)) for p in policies],
)
print(
    "Summary: the cutoff was chosen on the first half only. The following results use the fixed cutoff."
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for metric, label in (
    ("micro_precision", "Precision"),
    ("micro_recall", "Recall"),
    ("micro_f1", "F1"),
    ("coverage", "Coverage"),
):
    ax.plot(
        THRESHOLDS,
        [
            r["metrics"][metric] if r["metrics"][metric] is not None else np.nan
            for r in primary["threshold_trials"]
        ],
        marker=".",
        label=label,
    )
ax.axvline(primary["threshold"], color="gray", linestyle="--", label="Chosen cutoff")
ax.axhline(0.8, color="black", linestyle=":", label="80% precision/recall target")
ax.axhline(0.95, color="gray", linestyle=":", label="95% coverage target")
ax.set(
    xlabel="Minimum NLI score",
    ylabel="Fraction of tags or events",
    ylim=(0, 1.02),
    title="Choose a cutoff: 94 development examples with historical labels",
)
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)

## 7. Compare with rules and TF-IDF

We only load baseline and classical reports whose input fingerprints match this run. Predictions are aligned by event ID and rescored on these same 94 check examples. For classical models, the report must also use the same seed and validation division, so its chosen cutoff remains comparable. A missing matching report stops the comparison with instructions to rerun the earlier notebook.

The main table uses historical labels. The pilot table uses the reviewer’s 40 reviewed development decisions. Read precision, recall, and coverage together; all three project targets must hold for one policy. F1 helps compare their balance but does not replace the targets.

In [ ]:
baseline_path, baseline = matching_report(
    root / "reports/baselines",
    "baseline_version",
    source=source,
    selection=selection,
    fingerprints=expected_fingerprints,
)
classical_path, classical = matching_report(
    root / "reports/classical",
    "model_version",
    source=source,
    selection=selection,
    fingerprints=expected_fingerprints,
)
if (
    classical["random_seed"] != RANDOM_SEED
    or classical["threshold_selection_records"] != len(tune_index)
    or classical["group_heldout_check_records"] != len(check_index)
):
    raise ValueError("Classical report used a different validation division")
comparison = []
for origin, runs in (("baseline", baseline["runs"]), ("classical", classical["policies"])):
    for run in runs:
        name = run.get("candidate", run.get("model"))
        if origin == "baseline" and name not in {"most_common", "keywords"}:
            continue
        predictions = aligned_predictions(run["predictions"], validation)
        check = score_indices(historical, predictions, check_index)
        if origin == "classical" and check != run["group_heldout_check"]:
            raise ValueError("Classical check metrics do not match this event subset")
        comparison.append(
            {
                "candidate": name,
                "policy": run["policy"],
                "check": check,
                "pilot": multilabel_metrics(
                    [reviewed_pilot[validation[i]["external_id"]] for i in pilot_index],
                    [predictions[i] for i in pilot_index],
                    tags,
                ),
            }
        )
comparison += [
    {
        "candidate": "mdeberta_nli",
        "policy": p["policy"],
        "check": p["group_heldout_check"],
        "pilot": p["reviewed_pilot"],
    }
    for p in policies
]


columns = ["Candidate / policy", "N", "Precision", "Recall", "F1", "Coverage", ">2 tags"]
print("Same 94 check examples; historical reference tags")
show(
    columns,
    [
        comparison_row(f"{r['candidate']} / {r['policy']}", r["check"], include_exact_match=False)
        for r in comparison
    ],
)
print("Reviewed pilot: familiar development data, not the final test")
show(
    columns,
    [
        comparison_row(f"{r['candidate']} / {r['policy']}", r["pilot"], include_exact_match=False)
        for r in comparison
    ],
)

m = primary["group_heldout_check"]
meets_quality = all(
    m[k] is not None and m[k] >= target
    for k, target in (("micro_precision", 0.8), ("micro_recall", 0.8), ("coverage", 0.95))
)
print(
    f"NLI summary: precision {percent(m['micro_precision'])}, recall {percent(m['micro_recall'])}, "
    f"F1 {percent(m['micro_f1'])}, coverage {percent(m['coverage'])}."
)
print("Meets all three development quality targets:", meets_quality)
for r in comparison:
    if r["policy"] == "include_cutoff_ties" and r["candidate"] in {"keywords", "word_plus_char"}:
        delta = 100 * (m["micro_f1"] - r["check"]["micro_f1"])
        print(f"NLI F1 change versus {r['candidate']}: {delta:+.1f} percentage points.")

## 8. Inspect weak tags and resource costs

A single overall score can hide weak topics. This table shows how many examples support each tag, how many it finds, and how many it misses. An unsupported tag has no recall estimate. The first eight disagreements by event ID are shown privately for diagnosis; they are not a reason to change reference labels to match the model.

Inference timing includes all 12 title/tag pairs and tokenization for one event. It excludes downloading, loading, and the synthetic warm-up, which are reported separately. Cached event times come from the run that first computed them. TF-IDF's batch prediction time is a throughput measurement; it is not directly the same as this per-event latency. No API spend does not mean no compute cost.

Section 9 also writes a private Markdown review containing every disagreement in the 94-example check subset. Each entry shows the original title, historical tags, NLI suggestions, relevant scores, and space for your decision. These are disagreements, not proven model errors; review both sides against the rubric before changing labels. No cutoff-selection or final-test examples enter that export.

In [ ]:
show(
    ["Tag", "Reference events", "Correct", "Extra", "Missed", "Precision", "Recall"],
    [
        (
            tag,
            item["support"],
            item["tp"],
            item["fp"],
            item["fn"],
            percent(item["precision"]),
            percent(item["recall"]),
        )
        for tag, item in primary["group_heldout_check"]["per_tag"].items()
    ],
)
predictions = {item["external_id"]: item["predicted"] for item in primary["predictions"]}
mistakes = []
for i in sorted(check_index, key=lambda i: validation[i]["external_id"]):
    row = validation[i]
    actual, expected = set(predictions[row["external_id"]]), set(row["tag_slugs"])
    if actual != expected:
        mistakes.append(
            (
                row["external_id"],
                row["title"],
                ", ".join(sorted(expected - actual)),
                ", ".join(sorted(actual - expected)),
            )
        )
show(["ID", "Original title", "Missed", "Extra"], mistakes[:8])
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_process_mb = peak / (1_000_000 if sys.platform == "darwin" else 1_000)
show(
    ["Resource", "Measured value"],
    [
        ("Peak notebook process RSS, MB (not whole container)", round(peak_process_mb, 1)),
        ("Recorded median event inference, seconds", round(float(np.median(event_seconds)), 3)),
        (
            "Recorded p95 event inference, seconds",
            round(float(np.percentile(event_seconds, 95)), 3),
        ),
        ("Recorded maximum event inference, seconds", round(float(event_seconds.max()), 3)),
        ("Events under five minutes (inference only)", int((event_seconds < 300).sum())),
        ("API spend, USD", 0),
        ("Local compute cost priced", "No"),
    ],
)
print(
    "Summary: use tag support and the reviewed pilot to interpret quality; use actual inference times, not cache time, for speed."
)

## 9. Save the experiment and review the conclusion

The private report records the exact model revision, tag sentences, rubric and input fingerprints, scores, neutral outputs, cutoff trials, comparisons, timing, hardware, and cache use. It contains no final-test predictions. The cache lets us rerun reporting without repeating unchanged inference; both cache and report stay outside Git.

**Review checklist:** Check the synthetic example, the 94/94 division, the chosen cutoff, the historical and pilot tables, and the speed/memory table. Decide whether this candidate adds enough value to carry forward. The recorded conclusion below will summarize the completed run; the live tables remain the source for later reruns. Review accepts or corrects the conclusion before any final evaluation or service work.

In [ ]:
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
notebook_path = root / "notebooks/06_semantic_models.ipynb"
notebook_source_sha256 = notebook_sha256(notebook_path)
report = {
    "run_id": run_id,
    "candidate_version": "mdeberta-nli-v1",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "license": MODEL_LICENSE,
    "model_files_bytes": model_files_bytes,
    "checkpoint_sha256": EXPECTED_WEIGHT_SHA256,
    "tokenizer_sha256": EXPECTED_TOKENIZER_SHA256,
    "notebook_source_sha256": notebook_source_sha256,
    "uv_lock_sha256": file_sha256(root / "uv.lock"),
    "snapshot_sha256": source["content_sha256"],
    "taxonomy_version": source["taxonomy_version"],
    "selection_sha256": selection["selection_sha256"],
    "fingerprints": expected_fingerprints,
    "inference_settings": inference_settings,
    "settings_sha256": settings_sha256,
    "hypotheses_sha256": hypotheses_sha256,
    "threshold_grid": THRESHOLDS,
    "random_seed": RANDOM_SEED,
    "training_records_used_to_fit": 0,
    "validation_records": len(validation),
    "final_test_records_scored": 0,
    "threshold_selection_ids": [validation[i]["external_id"] for i in tune_index],
    "check_ids": [validation[i]["external_id"] for i in check_index],
    "policies": policies,
    "comparison": comparison,
    "comparison_sources": {
        str(p.relative_to(root)): file_sha256(p) for p in (baseline_path, classical_path)
    },
    "snapshot_seconds": snapshot_seconds,
    "model_load_seconds": model_load_seconds,
    "warmup_seconds": warmup_seconds,
    "scoring_wall_seconds": scoring_wall_seconds,
    "cache_hits": cache_hits,
    "fresh_events": len(validation) - cache_hits,
    "recorded_prediction_seconds": float(event_seconds.sum()),
    "recorded_median_event_seconds": float(np.median(event_seconds)),
    "recorded_p95_event_seconds": float(np.percentile(event_seconds, 95)),
    "peak_notebook_process_rss_mb": peak_process_mb,
    "cpu_architecture": platform.machine(),
    "cpu_threads": CPU_THREADS,
    "api_cost_usd": 0,
    "local_compute_cost_measured": False,
    "events": [
        {"external_id": row["external_id"], **event}
        for row, event in zip(validation, scored_events)
    ],
    "limitations": [
        "Historical validation labels can be wrong; the reviewed pilot is small and already familiar.",
        "Previously inspected development groups do not provide a fresh final-test estimate.",
        "Fixed short hypotheses cannot enforce all rubric exclusions or overlaps.",
        "NLI support-versus-contradiction scores are uncalibrated and omit neutral from normalization.",
        "No training on DemoLogic labels; model weights were pretrained and tag definitions are supplied.",
        "Latency excludes download/load/warm-up; cached records retain their original measured inference times.",
        "Rare or unsupported tags limit per-tag conclusions; final-test labels were not used for fitting, tuning or scoring.",
    ],
}
report_path = save_private_json(root / "reports/semantic" / f"{run_id}.json", report)
print("Private semantic report:", report_path)
review_path = write_nli_review(
    root / "reports/semantic-review" / f"{run_id}.md", report, validation
)
print("Private disagreement review:", review_path)

## 10. What we learned — 28 September 2026

**This fixed semantic candidate did not improve our tag suggestions.** The cutoff chosen on the first 94 examples was **0.35**. On the other 94 examples, it reached **30.2% precision, 29.9% recall, 30.0% F1, and 92.6% coverage** against historical labels. Production-derived regex reached **64.2% F1**, and word + character TF-IDF reached **60.3% F1** on those same examples.

**More suggestions did not mean better suggestions.** NLI covered more events than those two candidates, but proposed `human-rights` for 85 of the 94 check events; only 23 matched that historical reference tag. Its short tag descriptions may be too broad, and the score ignores neutral when comparing support with contradiction. These are possible explanations to investigate, not proven causes. Historical labels can also be wrong.

**The reviewed pilot points in the same direction.** On its 40 familiar development examples, NLI reached **38.2% precision, 29.2% recall, 33.1% F1, and 95.0% coverage**. This is useful supporting evidence, but neither this pilot nor the historical-label check estimates final-test performance. Strict-two and extra-cutoff-ties policies produced identical NLI results in this run.

**Local inference was practical, but quality was the limitation.** The first run took about **614 seconds for 188 events**, with **3.20 seconds median** and **3.86 seconds at the 95th percentile** per event. Peak notebook-process memory was about **2.24 GB**. These measurements used four CPU threads and exclude download, model loading, and warm-up. No titles were truncated. API spend was **$0**; local compute still has a cost that we have not priced. Rerunning from cache is faster, but is not a new inference-speed measurement.

**Recommendation for the reviewer to review:** Keep the experiment as evidence, but do not select this model-and-description setup for the service. It misses the agreed precision, recall, and coverage targets. This result does not rule out other semantic models or wording. Any follow-up should state a new question and preserve this run for comparison; do not use final-test answers to improve it.

These are dated checkpoint results. If inputs or settings change, use the freshly generated tables above and record a new comparison rather than silently replacing this conclusion.

## 11. Does the human-rights description crowd out other tags?

**Two follow-up experiments, requested by the reviewer on 29 September.** Keep the original run above unchanged. First rewrite only the human-rights hypothesis to ask for an explicit, central human/civil-rights demand. The approved rubric is unchanged; this is a different explanation to the model, not a new label definition. Second remove human-rights from the candidate list and let the next qualifying tag enter the top-two recommendations.

NLI scores each title/tag pair independently. We therefore reuse the other 11 scores exactly. The wording experiment needs only one new pair per validation event; removing a tag needs no inference. Neither experiment uses final-test data or changes historical labels.

For each variant, choose the cutoff on the same 94 tuning examples using **12-tag micro F1**, then check it on the other 94. Also show results at the original cutoff, so changes in wording are distinguishable from cutoff changes. Report the removal experiment against both the original 12-tag task and an 11-tag task with human-rights removed from **both references and predictions**. Keep all events, including those with no remaining reference tags; coverage means receiving any remaining suggestion. Compare with the original predictions under that same 11-tag scoring rule.

These validation cases have already been inspected. Improvements are exploratory, not independent confirmation. We will inspect per-tag precision/recall and changes in the suggested tags; two screenshots alone do not define success.

In [ ]:
HR_HYPOTHESIS = (
    "Diese Versammlung fordert ausdrücklich den Schutz der Menschen- oder Bürgerrechte "
    "von Menschen, etwa Schutz vor Rassismus oder Diskriminierung, gleiche Rechte, "
    "die Rechte von Migranten oder Menschen mit Behinderungen. "
    "Diese Rechte sind ein zentrales Anliegen der Versammlung."
)
hr_index = tags.index("human-rights")
show(
    ["Variant", "Human-rights hypothesis"],
    [
        ("Original", HYPOTHESES["human-rights"]),
        ("Clearer central-rights demand", HR_HYPOTHESIS),
        ("Excluded tag", "No human-rights suggestion; all other scores stay unchanged"),
    ],
)

# This cache is separate from the original model run and preserves its results.
hr_settings = {
    "base_settings_sha256": settings_sha256,
    "hypothesis": HR_HYPOTHESIS,
    "candidate": "human-rights-wording-v1",
    "batch_size": 1,
}
hr_cache_dir = root / "reports/semantic-hr-cache" / fingerprint(hr_settings)
hr_results = []
hr_cache_hits = 0
for index, row in enumerate(validation):
    identity = {
        "settings": fingerprint(hr_settings),
        "external_id": row["external_id"],
        "title_sha256": fingerprint(row["title"]),
    }
    path = hr_cache_dir / f"{fingerprint(identity)}.json"
    if path.exists():
        result = json.loads(path.read_text())
        if result["identity"] != identity:
            raise ValueError("Human-rights experiment cache mismatch")
        hr_cache_hits += 1
    else:
        started = perf_counter()
        inputs = tokenizer(
            row["title"],
            HR_HYPOTHESIS,
            truncation="only_first",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        with torch.inference_mode():
            logits = model(**inputs).logits.cpu().numpy()
        support, neutral = nli_tag_scores(logits, label_ids)
        result = {
            "identity": identity,
            "score": float(support[0]),
            "neutral": float(neutral[0]),
            "seconds": perf_counter() - started,
        }
        save_private_json(path, result)
    if not all(np.isfinite(result[k]) and 0 <= result[k] <= 1 for k in ["score", "neutral"]):
        raise ValueError("Invalid human-rights experiment score")
    hr_results.append(result)
    if (index + 1) % 40 == 0:
        print(f"Human-rights wording: {index + 1}/{len(validation)}", flush=True)

wording_scores = scores.copy()
wording_scores[:, hr_index] = [r["score"] for r in hr_results]
excluded_scores = scores.copy()
excluded_scores[:, hr_index] = -np.inf  # Never qualifies; other tags may fill the available slot.
hr_variants = {
    "original": scores,
    "clearer_wording": wording_scores,
    "exclude_human_rights": excluded_scores,
}
print("New wording cache hits / fresh calls:", hr_cache_hits, "/", len(validation) - hr_cache_hits)

In [ ]:
remaining_tags = [tag for tag in tags if tag != "human-rights"]


def hr_metrics(reference, predicted, indices, *, remaining_only=False):
    """Evaluate identical event indices; optionally remove HR from both sides, not events."""
    truth = [reference[i] for i in indices]
    guesses = [predicted[i] for i in indices]
    if remaining_only:
        truth = [[t for t in row if t != "human-rights"] for row in truth]
        guesses = [[t for t in row if t != "human-rights"] for row in guesses]
    return multilabel_metrics(truth, guesses, remaining_tags if remaining_only else tags)


hr_runs = []
for name, values in hr_variants.items():
    trials = []
    for cutoff in THRESHOLDS:
        predictions = [select_tags(row, cutoff, include_ties=True) for row in values]
        trials.append(
            {"threshold": cutoff, "metrics": hr_metrics(historical, predictions, tune_index)}
        )
    chosen_trial = max(
        trials,
        key=lambda r: (
            r["metrics"]["micro_f1"] if r["metrics"]["micro_f1"] is not None else -1,
            r["threshold"],
        ),
    )
    for mode, cutoff in [
        ("original_cutoff", primary["threshold"]),
        ("retuned_on_94", chosen_trial["threshold"]),
    ]:
        predicted = [select_tags(row, cutoff, include_ties=True) for row in values]
        hr_runs.append(
            {
                "variant": name,
                "mode": mode,
                "threshold": cutoff,
                "check_12": hr_metrics(historical, predicted, check_index),
                "check_11": hr_metrics(historical, predicted, check_index, remaining_only=True),
                "pilot": multilabel_metrics(
                    [reviewed_pilot[validation[i]["external_id"]] for i in pilot_index],
                    [predicted[i] for i in pilot_index],
                    tags,
                ),
                "predictions": [
                    {"external_id": row["external_id"], "predicted": pred}
                    for row, pred in zip(validation, predicted)
                ],
                "threshold_trials": trials,
            }
        )

for task in ["check_12", "check_11"]:
    print(
        "Original 12-tag task"
        if task == "check_12"
        else "Same events, human-rights removed from both sides"
    )
    show(
        ["Variant / cutoff mode", "Cutoff", "Precision", "Recall", "F1", "Coverage"],
        [
            (
                f"{r['variant']} / {r['mode']}",
                r["threshold"],
                *[
                    percent(r[task][key])
                    for key in ["micro_precision", "micro_recall", "micro_f1", "coverage"]
                ],
            )
            for r in hr_runs
        ],
    )
show(
    ["Variant / mode", "HR correct", "HR extra", "HR missed", "HR precision", "HR recall"],
    [
        (
            f"{r['variant']} / {r['mode']}",
            *[r["check_12"]["per_tag"]["human-rights"][k] for k in ["tp", "fp", "fn"]],
            *[
                percent(r["check_12"]["per_tag"]["human-rights"][k])
                for k in ["precision", "recall"]
            ],
        )
        for r in hr_runs
    ],
)
# Per-tag tables show whether the remaining topics improve, not only a combined score.
for r in hr_runs:
    if r["mode"] != "retuned_on_94":
        continue
    print(r["variant"], "— remaining tag results")
    show(
        ["Tag", "Reference count", "Precision", "Recall", "F1"],
        [
            (tag, m["support"], percent(m["precision"]), percent(m["recall"]), percent(m["f1"]))
            for tag, m in r["check_11"]["per_tag"].items()
        ],
    )

hr_report_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
hr_report = {
    "run_id": hr_report_id,
    "experiment": "human-rights-followup-v1",
    "base_report": str(report_path.relative_to(root)),
    "base_report_sha256": file_sha256(report_path),
    "notebook_source_sha256": notebook_sha256(root / "notebooks/06_semantic_models.ipynb"),
    "settings": hr_settings,
    "original_hypothesis": HYPOTHESES["human-rights"],
    "cache_hits": hr_cache_hits,
    "api_cost_usd": 0,
    "recorded_wording_seconds": sum(r["seconds"] for r in hr_results),
    "final_test_records_scored": 0,
    "runs": hr_runs,
    "wording_predictions": [
        {"external_id": row["external_id"], **result} for row, result in zip(validation, hr_results)
    ],
}
hr_report_path = save_private_json(root / "reports/semantic-hr" / f"{hr_report_id}.json", hr_report)
print("Private follow-up report:", hr_report_path)

## 12. Follow-up findings — 29 September 2026

**The two tests produced a useful diagnosis, but no convincing replacement for the baseline.** All three variants selected the same 0.35 cutoff on the 94 tuning examples. The following results use the other 94 events and historical references, not the final test.

| Original 12-tag task | Precision | Recall | F1 | Coverage |
|---|---:|---:|---:|---:|
| Original wording | 30.2% | 29.9% | 30.0% | 92.6% |
| Revised human-rights wording | 30.6% | 32.3% | 31.4% | 100.0% |
| Exclude human-rights | 27.0% | 13.4% | 17.9% | 42.6% |

**Wording:** The revised sentence suggested human-rights for all 94 check events, compared with 85 before. Historical references contain that tag in only 26 events. Extra human-rights suggestions increased from 62 to 68; HR recall increased from 88.5% to 100%. The small F1 increase is a trade-off, not evidence that the description became more selective. This tests one rewrite; it does not establish that every wording change would fail. On the familiar reviewed pilot, overall F1 moved from 33.1% to 33.8%; this is also only a development diagnostic.

**Exclusion:** To inspect the other topics fairly, remove human-rights from both the original predictions and the reference labels, keeping all 94 events. That original 11-tag diagnostic gives 21.1% F1. Excluding the candidate before selecting suggestions gives 20.7% F1. Recall rises from 14.9% to 16.8%, but precision falls from 36.6% to 27.0%; coverage remains 42.6%. The freed slots add 22 suggestions, only two of which match additional historical reference tags. Removing human-rights does not improve the remaining scores or make this setup a reliable 11-tag classifier.

**What we learned:** Human-rights over-tagging is a problem, but it is not the only weakness of this fixed NLI setup. A narrower-sounding description is not automatically a better model instruction. Most events still receive no other qualifying tag at this cutoff. Historical reference errors remain possible; these experiments do not establish their correctness or diagnose the internal cause of the model's behavior.

**Recommendation for review:** Keep the original baseline and both follow-ups as evidence. Do not promote either variant or change the taxonomy. The bounded title-versus-time/location comparison has since been completed in Notebook 05. Its results are recorded there and in the Week 2 plan.

**Cost and reproducibility:** The wording variant scored 188 additional title/hypothesis pairs in about 67 seconds of recorded local inference. The other 11 tag scores were reused exactly; exclusion required no model call. API spend was $0. Settings, all cutoff trials, scores, and predictions are saved privately under `reports/semantic-hr/`; reruns use `reports/semantic-hr-cache/`. No final-test records or labels were used, and no dataset labels changed.


## 13. Small synthetic diagnostic: support or mostly neutral?

We created ten German titles before running this diagnostic: two each about animal welfare, cycling, wages, religious worship, and explicit human rights. These are invented examples, not production records. For each title, test the original human-rights description, the revised description, and the original description of the intended topic as a control.

Show the full three-class output: **support + contradiction + neutral = 1**. Beside it, show the current tag score, **support / (support + contradiction)**. A high neutral output can therefore coexist with a high tag score. Passing 0.35 here means passing the human-rights cutoff only; it is not a full top-two recommendation because we are not evaluating all 12 tags.

This small diagnostic can reveal behavior on these examples. It cannot establish ideological bias, general accuracy, or the cause of behavior learned during pretraining. No model, threshold, hypothesis, or dataset label is changed in response to these diagnostic answers.

In [ ]:
SYNTHETIC_CASES = [
    ("animal_1", "Für bessere Lebensbedingungen von Schweinen in der Tierhaltung", "environment"),
    ("animal_2", "Stoppt Tierversuche und schützt die Tiere", "environment"),
    ("cycling_1", "Mehr sichere Fahrradwege und Fahrradparkplätze", "mobility"),
    ("cycling_2", "Geschützte Radwege an gefährlichen Kreuzungen bauen", "mobility"),
    ("wages_1", "Höhere Löhne und ein neuer Tarifvertrag für Beschäftigte", "work"),
    ("wages_2", "Streik für bessere Arbeitsbedingungen und faire Bezahlung", "work"),
    ("worship_1", "Gemeinsames Gebet und christlicher Gottesdienst", "religion"),
    ("worship_2", "Versammlung zum gemeinsamen religiösen Gebet", "religion"),
    ("rights_1", "Für Menschenrechte und gegen rassistische Diskriminierung", "human-rights"),
    ("rights_2", "Für Meinungsfreiheit und gleiche Bürgerrechte für alle Menschen", "human-rights"),
]


def diagnostic_probabilities(logits, label_ids):
    """Return full NLI probabilities and the existing binary score for one title/hypothesis.

    Full probabilities share one denominator and sum to one. The binary tag score
    omits neutral. Stable softmax avoids overflow; nli_tag_scores validates logits.
    """
    binary, _ = nli_tag_scores(logits, label_ids)
    values = np.asarray(logits, dtype=np.float64)
    full = np.exp(values - values.max(axis=1, keepdims=True))
    full /= full.sum(axis=1, keepdims=True)
    return {
        "support": float(full[0, label_ids["entailment"]]),
        "contradiction": float(full[0, label_ids["contradiction"]]),
        "neutral": float(full[0, label_ids["neutral"]]),
        "tag_score": float(binary[0]),
    }


diagnostic_rows = []
started = perf_counter()
for case_id, title, expected_tag in SYNTHETIC_CASES:
    for variant, hypothesis in [
        ("original_hr", HYPOTHESES["human-rights"]),
        ("revised_hr", HR_HYPOTHESIS),
        ("intended_topic_control", HYPOTHESES[expected_tag]),
    ]:
        inputs = tokenizer(
            title, hypothesis, truncation="only_first", max_length=MAX_LENGTH, return_tensors="pt"
        )
        with torch.inference_mode():
            logits = model(**inputs).logits.cpu().numpy()
        probabilities = diagnostic_probabilities(logits, label_ids)
        diagnostic_rows.append(
            {
                "case_id": case_id,
                "title": title,
                "expected_tag": expected_tag,
                "variant": variant,
                "hypothesis": hypothesis,
                **probabilities,
                "passes_cutoff": probabilities["tag_score"] >= primary["threshold"],
            }
        )
diagnostic_seconds = perf_counter() - started
show(
    ["Case", "Intended topic", "Synthetic title"],
    [(case, tag, title) for case, title, tag in SYNTHETIC_CASES],
)
show(
    ["Case", "Hypothesis", "Support", "Contradiction", "Neutral", "Tag score", "Passes cutoff"],
    [
        (
            r["case_id"],
            r["variant"],
            *[round(r[k], 4) for k in ["support", "contradiction", "neutral", "tag_score"]],
            r["passes_cutoff"],
        )
        for r in diagnostic_rows
    ],
)
diagnostic_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
diagnostic_report = {
    "run_id": diagnostic_id,
    "experiment": "synthetic-nli-neutral-diagnostic-v1",
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "base_settings_sha256": settings_sha256,
    "cutoff": primary["threshold"],
    "device": "cpu",
    "dtype": "float32",
    "batch_size": 1,
    "inference_seconds": diagnostic_seconds,
    "api_cost_usd": 0,
    "rows": diagnostic_rows,
    "notebook_source_sha256": notebook_sha256(root / "notebooks/06_semantic_models.ipynb"),
    "limitations": [
        "Ten invented titles; not a representative evaluation or proof of ideological bias.",
        "Per-pair cutoff check, not the complete twelve-tag recommendation policy.",
        "No labels, cutoff or model settings changed based on these results.",
    ],
}
diagnostic_path = save_private_json(
    root / "reports/semantic-diagnostic" / f"{diagnostic_id}.json", diagnostic_report
)
print("Diagnostic report:", diagnostic_path)
print("Seconds for 30 title/hypothesis pairs:", round(diagnostic_seconds, 2))

## 14. Diagnostic finding — mostly neutral can still pass the cutoff

**All eight synthetic titles about other topics passed the human-rights cutoff under both descriptions. In every case, neutral was the largest of the three model outputs.** This directly shows how our binary scoring rule can accept weak support on these examples. It is not evidence of ideological bias.

| Invented title / description | Full support | Full neutral | Current tag score |
|---|---:|---:|---:|
| Better living conditions for pigs / original HR | 0.0765 | 0.9166 | 0.9170 |
| Better living conditions for pigs / revised HR | 0.0047 | 0.9951 | 0.9778 |
| Higher wages and a new collective agreement / original HR | 0.1340 | 0.8534 | 0.9141 |
| Higher wages and a new collective agreement / revised HR | 0.0052 | 0.9947 | 0.9763 |
| Human rights and opposition to racist discrimination / original HR | 0.1012 | 0.8967 | 0.9798 |
| Human rights and opposition to racist discrimination / revised HR | 0.0076 | 0.9923 | 0.9844 |

The table paraphrases the German titles defined in section 13; values are rounded. The full output table above includes contradiction and all ten cases. The revised hypothesis has approximately 0.99 or higher neutral output for all ten titles, including the two intended human-rights examples. Tiny support still exceeds even smaller contradiction, producing a high binary score. This helps explain why more restrictive wording did not reduce suggestions.

**The controls matter:** The intended mobility, work, and religion descriptions gave their clear examples strong full support (approximately 0.765–0.997). Animal-welfare controls were less consistent. The model can recognize some clear topics; these ten examples do not support a claim that it fails every category.

**What this does and does not establish:** The current scoring rule demonstrably turns mostly neutral output into qualifying HR scores on this small set. Wording and model behavior also matter: even explicit human-rights titles remain mostly neutral for these hypotheses. We cannot isolate a pretraining cause or claim political bias. Using full support instead would produce different values, but applying the same 0.35 cutoff would reject the two positive HR controls as well. Any new scoring policy would need its own development evaluation and cutoff selection.

**Decision:** Keep this diagnostic as evidence; do not change scoring, the rubric, or labels based on ten invented titles. It took about 9.2 seconds for 30 pairs, with $0 API spend. The complete result is in `reports/semantic-diagnostic/20260929T104830662646Z.json`. No final-test examples were used.


## 15. Does language or model choice change these examples?

Compare three setups on the **same ten invented titles**: current mDeBERTa with German text; the same model with English translations; and German-specific GBERT with German text. Translate both titles and hypotheses. The translations below were written by Codex and fixed before this run; no translation API is used. They are visible for review. This tests these particular translations, not a production translation pipeline.

Keep the original HR hypothesis, revised HR hypothesis, and intended-topic control from section 13. Inspect full support, contradiction and neutral alongside the binary score. The old **0.35 cutoff is only a shared diagnostic reference**, not a validated threshold for English or GBERT. These are isolated pairs, not all twelve tags or complete recommendations. Do not compare their pass counts as benchmark accuracy.

German candidate: [Sahajtomar/German_Zeroshot](https://huggingface.co/Sahajtomar/German_Zeroshot/blob/d5b0a26665b8538bcb3faa1e63a634cca4c8ee1b/README.md), GBERT Large fine-tuned on German XNLI, pinned at `d5b0a26665b8538bcb3faa1e63a634cca4c8ee1b`. The card does not declare a model license; it is an exploratory local candidate, not cleared for redistribution or service deployment. Record file hashes and load only tensor weights with PyTorch's restricted weights-only loader. The larger model changes architecture and training as well as language specialization, so a difference cannot be attributed to German specialization alone.

No training, label edits, private event translation, or final-test scoring occurs here. A promising result supports a later validation experiment with its own cutoff selection; ten hand-written titles cannot establish general quality.

In [ ]:
ENGLISH_TITLES = {
    "animal_1": "For better living conditions for pigs in animal farming",
    "animal_2": "Stop animal testing and protect animals",
    "cycling_1": "More safe bicycle lanes and bicycle parking spaces",
    "cycling_2": "Build protected bicycle lanes at dangerous intersections",
    "wages_1": "Higher wages and a new collective agreement for employees",
    "wages_2": "Strike for better working conditions and fair pay",
    "worship_1": "Joint prayer and Christian worship service",
    "worship_2": "Assembly for joint religious prayer",
    "rights_1": "For human rights and against racist discrimination",
    "rights_2": "For freedom of expression and equal civil rights for all people",
}
ENGLISH_HYPOTHESES = {
    "environment": "The central topic of this assembly is climate, the energy transition, nature conservation, animal protection or environmental protection.",
    "mobility": "The central topic of this assembly is traffic, cycling, walking, local public transport or safe streets.",
    "work": "The central topic of this assembly is work, wages, working conditions, strikes, trade unions or collective bargaining.",
    "religion": "The central topic of this assembly is religion, faith, religious communities, prayers, worship services or religious institutions.",
    "human-rights": "The central topic of this assembly is human rights, civil rights, anti-racism, migration, equal rights, accessibility or inclusion.",
}
ENGLISH_REVISED_HR = (
    "This assembly explicitly calls for the protection of people's human or civil rights, "
    "such as protection against racism or discrimination, equal rights, "
    "the rights of migrants or people with disabilities. "
    "These rights are a central concern of the assembly."
)
if set(ENGLISH_TITLES) != {case for case, _, _ in SYNTHETIC_CASES}:
    raise ValueError("Translate every synthetic case exactly once")
show(
    ["Case", "German title", "English translation"],
    [(case, title, ENGLISH_TITLES[case]) for case, title, _ in SYNTHETIC_CASES],
)
show(
    ["Tag", "English hypothesis"],
    list(ENGLISH_HYPOTHESES.items()) + [("revised human-rights", ENGLISH_REVISED_HR)],
)

In [ ]:
GERMAN_MODEL_ID = "Sahajtomar/German_Zeroshot"
GERMAN_REVISION = "d5b0a26665b8538bcb3faa1e63a634cca4c8ee1b"
GERMAN_FILES = [
    "config.json",
    "pytorch_model.bin",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "vocab.txt",
    "README.md",
]
started = perf_counter()
german_dir = Path(
    snapshot_download(
        repo_id=GERMAN_MODEL_ID,
        revision=GERMAN_REVISION,
        allow_patterns=GERMAN_FILES,
        cache_dir=str(root / "artifacts/hf-cache"),
    )
)
german_file_hashes = {name: file_sha256(german_dir / name) for name in GERMAN_FILES}
german_tokenizer = AutoTokenizer.from_pretrained(
    german_dir,
    use_fast=False,
    local_files_only=True,
    trust_remote_code=False,
)
german_model = (
    AutoModelForSequenceClassification.from_pretrained(
        german_dir,
        local_files_only=True,
        trust_remote_code=False,
        use_safetensors=False,
        weights_only=True,
        dtype=torch.float32,
    )
    .to("cpu")
    .eval()
)
german_label_ids = {name.lower(): int(i) for name, i in german_model.config.label2id.items()}
if set(german_label_ids) != {"entailment", "neutral", "contradiction"}:
    raise ValueError("German model must declare its NLI label order")
german_load_seconds = perf_counter() - started
print("German snapshot verification and loading seconds:", round(german_load_seconds, 2))
print(
    "Selected German model files, MB:",
    round(sum((german_dir / name).stat().st_size for name in GERMAN_FILES) / 1_000_000, 1),
)

In [ ]:
language_rows = []
language_timings = {}
for setup, candidate_model, candidate_tokenizer, candidate_labels, language in [
    ("mdeberta_de", model, tokenizer, label_ids, "de"),
    ("mdeberta_en", model, tokenizer, label_ids, "en"),
    ("gbert_de", german_model, german_tokenizer, german_label_ids, "de"),
]:
    started = perf_counter()
    for case_id, german_title, intended_tag in SYNTHETIC_CASES:
        title = ENGLISH_TITLES[case_id] if language == "en" else german_title
        descriptions = ENGLISH_HYPOTHESES if language == "en" else HYPOTHESES
        revised = ENGLISH_REVISED_HR if language == "en" else HR_HYPOTHESIS
        for variant, hypothesis in [
            ("original_hr", descriptions["human-rights"]),
            ("revised_hr", revised),
            ("intended_topic_control", descriptions[intended_tag]),
        ]:
            inputs = candidate_tokenizer(
                title,
                hypothesis,
                truncation="only_first",
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            with torch.inference_mode():
                logits = candidate_model(**inputs).logits.cpu().numpy()
            probabilities = diagnostic_probabilities(logits, candidate_labels)
            language_rows.append(
                {
                    "setup": setup,
                    "case_id": case_id,
                    "title": title,
                    "expected_tag": intended_tag,
                    "variant": variant,
                    "hypothesis": hypothesis,
                    **probabilities,
                    "passes_reference_cutoff": probabilities["tag_score"] >= 0.35,
                }
            )
    language_timings[setup] = perf_counter() - started
show(
    ["Setup", "Case", "Hypothesis", "Support", "Contradiction", "Neutral", "Binary score"],
    [
        (
            r["setup"],
            r["case_id"],
            r["variant"],
            *[round(r[key], 4) for key in ["support", "contradiction", "neutral", "tag_score"]],
        )
        for r in language_rows
    ],
)
# Count only the eight non-HR controls for unwanted HR cutoff passes.
show(
    ["Setup", "Other-topic HR passes: original / revised", "Intended-topic passes / 10", "Seconds"],
    [
        (
            setup,
            " / ".join(
                str(
                    sum(
                        r["passes_reference_cutoff"]
                        for r in language_rows
                        if r["setup"] == setup
                        and r["variant"] == variant
                        and r["expected_tag"] != "human-rights"
                    )
                )
                for variant in ["original_hr", "revised_hr"]
            ),
            sum(
                r["passes_reference_cutoff"]
                for r in language_rows
                if r["setup"] == setup and r["variant"] == "intended_topic_control"
            ),
            round(seconds, 2),
        )
        for setup, seconds in language_timings.items()
    ],
)
language_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
language_report = {
    "run_id": language_id,
    "experiment": "synthetic-nli-language-v1",
    "models": {
        "mdeberta": {"id": MODEL_ID, "revision": MODEL_REVISION},
        "gbert": {
            "id": GERMAN_MODEL_ID,
            "revision": GERMAN_REVISION,
            "file_sha256": german_file_hashes,
            "license": "not declared in card",
        },
    },
    "translation_method": "Codex-authored fixed translations, visible for human review",
    "input_sha256": fingerprint(
        {
            "cases": SYNTHETIC_CASES,
            "german": HYPOTHESES,
            "revised": HR_HYPOTHESIS,
            "english_titles": ENGLISH_TITLES,
            "english_hypotheses": ENGLISH_HYPOTHESES,
            "english_revised": ENGLISH_REVISED_HR,
        }
    ),
    "device": "cpu",
    "dtype": "float32",
    "batch_size": 1,
    "threads": CPU_THREADS,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "reference_cutoff": 0.35,
    "inference_seconds": language_timings,
    "german_load_seconds": german_load_seconds,
    "api_cost_usd": 0,
    "final_test_records_scored": 0,
    "rows": language_rows,
    "notebook_source_sha256": notebook_sha256(root / "notebooks/06_semantic_models.ipynb"),
    "limitations": [
        "Ten synthetic examples, not benchmark accuracy",
        "Cutoff is not validated for new setups",
        "No full twelve-tag selection",
        "German model differs in architecture and training, not only language",
        "Timing includes first-call overhead; not steady-state event latency",
    ],
}
language_path = save_private_json(
    root / "reports/semantic-language" / f"{language_id}.json", language_report
)
print("Private language comparison:", language_path)

## 16. Language diagnostic findings

At the shared 0.35 reference cutoff, English translation fixed one animal-welfare miss but lost both cycling matches. German GBERT rejected all eight unwanted HR candidates **with the original description**, while keeping both explicit HR positives. With the revised description, it again accepted HR for all eight unrelated titles. The wording and the way we turn NLI outputs into tag scores matter across models.

| Setup | Unwanted HR passes: original / revised (of 8) | Intended-topic passes (of 10) |
|---|---:|---:|
| Current model, German | 8 / 8 | 9 |
| Current model, English | 7 / 8 | 8 |
| German GBERT, German | 0 / 8 | 9 |

For the invented pig-welfare title, the environment binary score was **0.286 in the current German setup, 0.704 in English, and 0.019 with German GBERT**. These values are not calibrated confidence estimates. GBERT's two positive HR controls were still mostly neutral (about 90% and 94%). Its apparent improvement does not eliminate the neutral-score concern.

**Decision:** Do not adopt translation as a general fix from this mixed result. GBERT with the original German descriptions warrants a bounded check on the existing validation split. Keep both wording variants in the diagnostic report; select the original descriptions for the larger run before seeing its results. This selection uses already examined synthetic examples, so subsequent results remain exploratory development evidence.

The 90 pairs took about 26 seconds of inference, with $0 API spend. GBERT's selected download is about 1.34 GB. Timings include first-call overhead and are not a steady-state benchmark. The full initial report is `reports/semantic-language/20260929T110229851060Z.json`.

## 17. Does the German candidate help on real development examples?

Use the original twelve German hypotheses and the same 188 validation representatives. Choose GBERT's own cutoff on the designated 94 tuning events, using the same grid and highest-micro-F1 rule as the original NLI experiment. Apply that choice to the other 94 events, and show the reviewed pilot separately. These check cases have already informed our exploration, so this is not an untouched final evaluation.

Score locally on CPU, float32, four pairs per batch. Record scores, neutral outputs, truncation and time in a separate cache. Cache hits preserve their original measurement times; they do not measure fresh inference. No label edits, translated private events, or final-test scoring occur. The original multilingual baseline remains intact.

In [ ]:
def score_german_title(title):
    """Score all original German hypotheses with GBERT, without reading reference tags.

    Return binary scores and full neutral outputs in the shared tag order.
    Measure tokenization and inference only; count pairs that require truncation.
    """
    started = perf_counter()
    descriptions = [HYPOTHESES[tag] for tag in tags]
    title_length = len(german_tokenizer.encode(title, add_special_tokens=False))
    truncated = sum(
        title_length
        + len(german_tokenizer.encode(text, add_special_tokens=False))
        + german_tokenizer.num_special_tokens_to_add(pair=True)
        > MAX_LENGTH
        for text in descriptions
    )
    values, neutral_values = [], []
    with torch.inference_mode():
        for start in range(0, len(tags), BATCH_SIZE):
            batch = descriptions[start : start + BATCH_SIZE]
            inputs = german_tokenizer(
                [title] * len(batch),
                batch,
                padding=True,
                truncation="only_first",
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            binary, neutral = nli_tag_scores(
                german_model(**inputs).logits.cpu().numpy(), german_label_ids
            )
            values.extend(binary.tolist())
            neutral_values.extend(neutral.tolist())
    return {
        "scores": values,
        "neutral_probabilities": neutral_values,
        "truncated_pairs": truncated,
        "prediction_seconds": perf_counter() - started,
    }


gbert_settings = {
    **inference_settings,
    "model_id": GERMAN_MODEL_ID,
    "revision": GERMAN_REVISION,
    "label_ids": german_label_ids,
    "file_sha256": german_file_hashes,
}
gbert_settings_hash = fingerprint(gbert_settings)
gbert_events = []
gbert_cache_hits = 0
for index, row in enumerate(validation):
    identity = {
        "settings_sha256": gbert_settings_hash,
        "external_id": row["external_id"],
        "title_sha256": fingerprint(row["title"]),
    }
    path = (
        root
        / "reports/semantic-gbert-cache"
        / gbert_settings_hash
        / f"{fingerprint(identity)}.json"
    )
    if path.exists():
        event = json.loads(path.read_text())
        if event.get("identity") != identity:
            raise ValueError("GBERT cache identity mismatch")
        gbert_cache_hits += 1
    else:
        event = {"identity": identity, **score_german_title(row["title"])}
        save_private_json(path, event)
    for key in ["scores", "neutral_probabilities"]:
        values = np.asarray(event[key])
        if (
            values.shape != (len(tags),)
            or not np.isfinite(values).all()
            or np.any((values < 0) | (values > 1))
        ):
            raise ValueError(f"Invalid GBERT cached {key}")
    gbert_events.append(event)
    if (index + 1) % 20 == 0 or index + 1 == len(validation):
        print(f"GBERT: scored or loaded {index + 1}/{len(validation)}", flush=True)

gbert_scores = np.asarray([r["scores"] for r in gbert_events])
gbert_policies = []
for ties in [True, False]:
    trials = []
    for cutoff in THRESHOLDS:
        predictions = [select_tags(row, cutoff, include_ties=ties) for row in gbert_scores]
        trials.append(
            {"threshold": cutoff, "metrics": score_indices(historical, predictions, tune_index)}
        )
    chosen = max(
        trials,
        key=lambda r: (
            r["metrics"]["micro_f1"] if r["metrics"]["micro_f1"] is not None else -1,
            r["threshold"],
        ),
    )
    predictions = [select_tags(row, chosen["threshold"], include_ties=ties) for row in gbert_scores]
    gbert_policies.append(
        {
            "policy": "include_cutoff_ties" if ties else "strict_two",
            "threshold": chosen["threshold"],
            "threshold_trials": trials,
            "tuning": chosen["metrics"],
            "check": score_indices(historical, predictions, check_index),
            "pilot": multilabel_metrics(
                [reviewed_pilot[validation[i]["external_id"]] for i in pilot_index],
                [predictions[i] for i in pilot_index],
                tags,
            ),
            "predictions": [
                {"external_id": row["external_id"], "predicted": pred}
                for row, pred in zip(validation, predictions)
            ],
        }
    )
gbert_primary = gbert_policies[0]
show(
    ["Candidate", "Cutoff", "Precision", "Recall", "F1", "Coverage"],
    [
        (
            name,
            cutoff,
            *[
                percent(metrics[key])
                for key in ["micro_precision", "micro_recall", "micro_f1", "coverage"]
            ],
        )
        for name, cutoff, metrics in [
            ("Original mDeBERTa: check", primary["threshold"], primary["group_heldout_check"]),
            ("German GBERT: check", gbert_primary["threshold"], gbert_primary["check"]),
            ("German GBERT: reviewed pilot", gbert_primary["threshold"], gbert_primary["pilot"]),
        ]
    ],
)
show(
    ["Tag", "Reference count", "Precision", "Recall", "F1"],
    [
        (tag, m["support"], percent(m["precision"]), percent(m["recall"]), percent(m["f1"]))
        for tag, m in gbert_primary["check"]["per_tag"].items()
    ],
)
gbert_times = np.asarray([r["prediction_seconds"] for r in gbert_events])
gbert_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
gbert_report = {
    "run_id": gbert_id,
    "experiment": "german-nli-validation-v1",
    "random_seed": RANDOM_SEED,
    "threshold_selection_ids": [validation[i]["external_id"] for i in tune_index],
    "check_ids": [validation[i]["external_id"] for i in check_index],
    "base_report": str(report_path.relative_to(root)),
    "base_report_sha256": file_sha256(report_path),
    "settings": gbert_settings,
    "fingerprints": expected_fingerprints,
    "policies": gbert_policies,
    "events": gbert_events,
    "cache_hits": gbert_cache_hits,
    "recorded_inference_seconds": float(gbert_times.sum()),
    "recorded_median_seconds": float(np.median(gbert_times)),
    "recorded_p95_seconds": float(np.percentile(gbert_times, 95)),
    "final_test_records_scored": 0,
    "api_cost_usd": 0,
    "notebook_source_sha256": notebook_sha256(root / "notebooks/06_semantic_models.ipynb"),
    "limitations": [
        "Exploratory: check examples previously inspected",
        "Historical references can be imperfect",
        "German candidate license not declared in model card; no deployment clearance",
    ],
}
gbert_report_path = save_private_json(
    root / "reports/semantic-gbert" / f"{gbert_id}.json", gbert_report
)
print("Private German validation report:", gbert_report_path)
print("Cache hits:", gbert_cache_hits, "/", len(validation))
print("Recorded inference seconds:", round(float(gbert_times.sum()), 2))

## 18. German candidate: real-data result and decision

**The German specialist did not improve overall F1.** Its cutoff was 0.05, selected by the same highest-micro-F1 rule on the 94 tuning examples. This is the lowest cutoff in the declared grid; we did not expand the search after seeing the check results. The original model selected 0.35. Different cutoffs are expected because scores from different models are not interchangeable probabilities.

| Same 94 check events; historical references | Precision | Recall | F1 | Coverage |
|---|---:|---:|---:|---:|
| Original multilingual NLI | 30.2% | 29.9% | 30.0% | 92.6% |
| German GBERT NLI | 25.6% | 34.6% | 29.4% | 96.8% |

**What changed:** GBERT suggested more tags that matched the references, but also more that did not. Its coverage passed the 95% target, while precision and recall stayed well below the 80% targets. Human-rights extra suggestions fell from 62 to 12, but omitted historical HR labels rose from 3 to 16. Solving one over-tagging problem did not solve the complete task. Strict-two and tied-second-place policies produced the same results in this run.

On the familiar 40-record reviewed pilot, GBERT achieved **28.4% precision, 29.2% recall, 28.8% F1 and 97.5% coverage**. This does not support promoting it either. Both the pilot and historical check set are development evidence; the final test remains untouched.

**Decision:** Keep the original NLI baseline, the translation diagnostic and this German comparison as evidence. Do not replace the selected tagging approach with either NLI setup, and do not start translating all production titles. The ten synthetic examples were useful for finding a candidate, but did not predict its overall quality. We can now move to the planned Notebook 05 context experiment without adding more NLI variants to this round.

**Cost and reproducibility:** GBERT scored 188 events, each against 12 hypotheses, in about **291 seconds** of recorded local inference. Median event time was **1.37 seconds**, and the 95th percentile was **2.45 seconds**; these exclude model loading and downloads. This is a separate run, not a controlled hardware-speed comparison with the original model. API spend was **$0**. Settings, all cutoff trials, scores, neutral outputs, predictions and input fingerprints are saved in `reports/semantic-gbert/20260929T111018666989Z.json`; reruns reuse `reports/semantic-gbert-cache/`. No reference labels changed.

**Reference-label review:** The 16 omitted historical HR labels are disagreements, not 16 confirmed errors. Some titles clearly support HR; others may have overly broad historical assignments or need context. Nine HR scores fell below the 0.05 cutoff; seven passed it but were displaced by two higher-ranked tags. Distinguish scoring, ranking and reference-label issues before drawing conclusions. Any approved label corrections must be versioned and applied consistently when rescoring every candidate. No labels or metrics have been changed by this review.
